In [ ]:
# --- ensure repo-root cwd so data/ paths resolve from anywhere ---
import os, sys
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src'))

# SoH method comparison on one vehicle — `MB7F8CLLFNJH48488` (Mahindra Treo)

This Treo appears in **both** sources, so we compare two SoH estimates for the *same physical
battery*:

1. **Coulomb counting** (intellicar) — real `current`, `Q = ∫I·dt` per continuous session;
   capacity = `|ΔQ| / (|ΔSoC|/100)` (Ah). Monthly capacity = **75th percentile** of session
   estimates (≥3 sessions) — the upper envelope, since data gaps make coulomb counting
   *under*-estimate, never over-estimate.
2. **Distance-per-SoC** (mahindra OEM feed) — no current; capacity proxied by pooled
   `Σ Δodometer / Σ Δsoc` during discharge (km per %SoC).

Each monthly series is smoothed (rolling median), normalised to an early-life baseline, then
forced **monotonic non-increasing**.

**Caveat:** monthly-sampled data limits precision, and the two feeds' data-rich periods for this
vehicle only partially overlap, so treat this as a method-level comparison, not a calibrated SoH.

In [ ]:
import numpy as np, pandas as pd
import pyarrow.dataset as ds, pyarrow.compute as pc
import matplotlib.pyplot as plt
VIN = "MB7F8CLLFNJH48488"

def robust_soh(monthly, valcol, smooth=3, base_n=3):
    g = monthly.sort_values("month").copy()
    g["smooth"] = g[valcol].rolling(smooth, min_periods=1, center=True).median()
    base = g["smooth"].iloc[:base_n].median()
    g["soh_raw"] = (100.0 * g["smooth"] / base).clip(upper=102)
    g["soh"] = g["soh_raw"].cummin()
    return g

## A. Coulomb counting (intellicar)

In [ ]:
GAP_S, MIN_ROWS, MIN_DSOC, MIN_SESS, CAP_BOUNDS = 300.0, 5, 3.0, 3, (40.0, 400.0)
ic = ds.dataset("data/mahindra/_archive/intellicar_oldest10", format="parquet").to_table(
        columns=["vin", "eventAt", "soc", "current"], filter=pc.field("vin") == VIN).to_pandas()
ic["t"] = pd.to_datetime(pd.to_numeric(ic["eventAt"], errors="coerce"), unit="ms")
for c in ["soc", "current"]:
    ic[c] = pd.to_numeric(ic[c], errors="coerce")
ic = ic.dropna(subset=["t", "soc", "current"]).query("0 <= soc <= 100").sort_values("t").reset_index(drop=True)
print(f"intellicar rows: {len(ic):,}")

dt = ic["t"].diff().dt.total_seconds().to_numpy()
newses = ~np.isfinite(dt) | (dt > GAP_S) | (dt <= 0)
ic["session"] = np.cumsum(newses)
cur = ic["current"].to_numpy()
ic["dQ"] = np.where(newses, 0.0, (cur + np.roll(cur, 1)) / 2.0) * np.where(newses, 0.0, dt) / 3600.0

s = ic.groupby("session").agg(soc0=("soc", "first"), soc1=("soc", "last"),
                              dQ=("dQ", "sum"), n=("soc", "size"), t=("t", "first"))
s["dSoC"] = s["soc1"] - s["soc0"]
s = s[(s["n"] >= MIN_ROWS) & (s["dSoC"].abs() >= MIN_DSOC)]
s["cap"] = s["dQ"].abs() / (s["dSoC"].abs() / 100.0)
s = s[s["cap"].between(*CAP_BOUNDS)]
s["month"] = s["t"].dt.to_period("M").dt.to_timestamp()
cc_month = (s.groupby("month").agg(capacity_ah=("cap", lambda x: np.percentile(x, 75)),
                                   n_sessions=("cap", "size")).reset_index())
cc_month = cc_month[cc_month["n_sessions"] >= MIN_SESS]
cc_soh = robust_soh(cc_month, "capacity_ah")
print(f"coulomb: {len(s)} sessions -> {len(cc_soh)} monthly points; "
      f"capacity median {cc_month['capacity_ah'].median():.0f} Ah")

## B. Distance-per-SoC (mahindra OEM feed)

In [ ]:
MAX_STEP_KM, MIN_DROP, KMSOC, MONTH_MIN_SOC = 60.0, 1.0, (0.2, 12.0), 15.0
mh = ds.dataset("data/mahindra/_archive/compare_vin_feed", format="parquet").to_table(
        columns=["vin", "eventAt", "soc", "odometer"]).to_pandas()
mh["t"] = pd.to_datetime(pd.to_numeric(mh["eventAt"], errors="coerce"), unit="ms")
for c in ["soc", "odometer"]:
    mh[c] = pd.to_numeric(mh[c], errors="coerce")
mh = mh.dropna(subset=["t", "soc", "odometer"]).query("0 < soc <= 100 and odometer > 0").sort_values("t")
print(f"mahindra rows: {len(mh):,}")

st = pd.DataFrame({"t": mh["t"].values, "d_odo": mh["odometer"].diff().values,
                   "soc_drop": (-mh["soc"].diff()).values}).dropna()
st = st[(st["soc_drop"] >= MIN_DROP) & (st["d_odo"] > 0) & (st["d_odo"] <= MAX_STEP_KM)]
st["km_per_soc"] = st["d_odo"] / st["soc_drop"]
st = st[st["km_per_soc"].between(*KMSOC)]
st["month"] = st["t"].dt.to_period("M").dt.to_timestamp()
mh_month = st.groupby("month").apply(
    lambda g: pd.Series({"km_per_soc": g["d_odo"].sum() / g["soc_drop"].sum(),
                         "tot_soc": g["soc_drop"].sum()})).reset_index()
mh_month = mh_month[mh_month["tot_soc"] >= MONTH_MIN_SOC]
mh_soh = robust_soh(mh_month, "km_per_soc")
print(f"distance-per-soc: {len(st)} steps -> {len(mh_soh)} monthly points")

## C. Overlay and compare

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cc_soh["month"], cc_soh["soh"], "-o", color="C0", label="Coulomb counting (intellicar)")
ax.plot(cc_soh["month"], cc_soh["soh_raw"], ":", color="C0", alpha=0.4)
ax.plot(mh_soh["month"], mh_soh["soh"], "-s", color="C1", label="Distance-per-SoC (mahindra feed)")
ax.plot(mh_soh["month"], mh_soh["soh_raw"], ":", color="C1", alpha=0.4)
if len(cc_soh) and len(mh_soh):
    lo = max(cc_soh["month"].min(), mh_soh["month"].min())
    hi = min(cc_soh["month"].max(), mh_soh["month"].max())
    if lo <= hi:
        ax.axvspan(lo, hi, color="grey", alpha=0.15, label="overlap window")
ax.set_title(f"SoH method comparison — {VIN} (Treo)\nsolid = smoothed+monotonic, dotted = smoothed raw")
ax.set_ylabel("SoH (%)"); ax.set_xlabel("Month"); ax.grid(alpha=0.3); ax.legend()
ax.set_ylim(60, 105)
plt.tight_layout(); plt.show()

In [ ]:
m = pd.merge(cc_soh[["month", "soh"]].rename(columns={"soh": "soh_coulomb"}),
             mh_soh[["month", "soh"]].rename(columns={"soh": "soh_dist_per_soc"}), on="month")
m["abs_diff"] = (m["soh_coulomb"] - m["soh_dist_per_soc"]).abs()
print(f"Direct overlap months: {len(m)}")
if len(m):
    print(m.assign(month=m["month"].dt.strftime("%Y-%m")).round(1).to_string(index=False))
    print(f"Mean |difference|: {m['abs_diff'].mean():.1f} pp")
def deg(g):
    x = (g["month"] - g["month"].iloc[0]).dt.days / 365.25
    return np.polyfit(x, g["soh"], 1)[0] if len(g) > 2 else np.nan
print(f"\nCoulomb:          covers {cc_soh['month'].min():%Y-%m}..{cc_soh['month'].max():%Y-%m}, "
      f"{deg(cc_soh):+.2f} %/yr, current {cc_soh['soh'].iloc[-1]:.1f}%")
print(f"Distance-per-SoC: covers {mh_soh['month'].min():%Y-%m}..{mh_soh['month'].max():%Y-%m}, "
      f"{deg(mh_soh):+.2f} %/yr, current {mh_soh['soh'].iloc[-1]:.1f}%")